In [5]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Base URL of the website
base_url = "http://www.mscepune.in/GCC/MAR40.aspx"

# Create folder to save PDFs
os.makedirs("pdfs", exist_ok=True)

# Get page content
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

# Find all <a> tags with .pdf links
pdf_links = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if ".pdf" in href.lower():
        full_url = urljoin(base_url, href)
        pdf_links.append(full_url)

print(f"Found {len(pdf_links)} PDF files")

# Download PDFs
for url in pdf_links:
    filename = url.split("/")[-1].replace(" ", "_")
    filepath = os.path.join("pdfs", filename)

    print(f"Downloading: {filename}")

    try:
        pdf_response = requests.get(url)
        with open(filepath, "wb") as f:
            f.write(pdf_response.content)
    except Exception as e:
        print(f"Failed: {url} -> {e}")

print("Download completed.")

Found 30 PDF files
Downloading: BATCH_-_(1101).pdf
Downloading: BATCH_-_(1102).pdf
Downloading: BATCH_-_(1103).pdf
Downloading: BATCH_-_(1104).pdf
Downloading: BATCH_-_(1105).pdf
Downloading: BATCH_-_(1201).pdf
Downloading: BATCH_-_(1202).pdf
Downloading: BATCH_-_(1203).pdf
Downloading: BATCH_-_(1204).pdf
Downloading: BATCH_-_(1205).pdf
Downloading: BATCH_-_(1301).pdf
Downloading: BATCH_-_(1302).pdf
Downloading: BATCH_-_(1303).pdf
Downloading: BATCH_-_(1304).pdf
Downloading: BATCH_-_(1305).pdf
Downloading: BATCH_-_(1401).pdf
Downloading: BATCH_-_(1402).pdf
Downloading: BATCH_-_(1403).pdf
Downloading: BATCH_-_(1404).pdf
Downloading: BATCH_-_(1405).pdf
Downloading: BATCH_-_(1501).pdf
Downloading: BATCH_-_(1502).pdf
Downloading: BATCH_-_(1503).pdf
Downloading: BATCH_-_(1504).pdf
Downloading: BATCH_-_(1505).pdf
Downloading: BATCH_-_(1601).pdf
Downloading: BATCH_-_(1602).pdf
Downloading: BATCH_-_(1603).pdf
Downloading: BATCH_-_(1604).pdf
Downloading: BATCH_-_(1605).pdf
Download completed.


In [6]:
!pip install pdfplumber pandas -q

In [7]:
import pdfplumber
import pandas as pd
import re
import os

def extract_pdf_to_rows(pdf_path):
    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += "\n" + text

    # Normalize text
    lines = [line.strip() for line in full_text.split("\n") if line.strip()]

    current = {}
    buffer = []

    for line in lines:
        # Detect new question row (starts with number)
        match = re.match(r"^(\d+)\s+(.*)", line)

        if match:
            # Save previous
            if current:
                current["Question"] = " ".join(buffer)
                rows.append(current)

            # Start new
            current = {
                "Sr.No": match.group(1),
                "Option A": "",
                "Option B": "",
                "Option C": "",
                "Option D": "",
                "Answer": ""
            }
            buffer = [match.group(2)]

        elif re.match(r"^[A-D]$", line):
            # Sometimes answer line isolated
            current["Answer"] = line

        elif re.match(r".*\s+[A-D]$", line):
            # Line ending with answer (e.g., "1 D")
            parts = line.rsplit(" ", 1)
            buffer.append(parts[0])
            current["Answer"] = parts[1]

        else:
            buffer.append(line)

    # Append last row
    if current:
        current["Question"] = " ".join(buffer)
        rows.append(current)

    return rows


def process_folder(pdf_folder, output_csv):
    all_data = []

    for file in os.listdir(pdf_folder):
        if file.endswith(".pdf"):
            path = os.path.join(pdf_folder, file)
            print(f"Processing: {file}")

            try:
                rows = extract_pdf_to_rows(path)
                for r in rows:
                    r["Source File"] = file
                all_data.extend(rows)
            except Exception as e:
                print(f"Error in {file}: {e}")

    df = pd.DataFrame(all_data)
    df.to_csv(output_csv, index=False)
    print(f"\nSaved CSV: {output_csv}")


# === RUN ===
process_folder("eng_30_pdfs", "output.csv")

Processing: BATCH_-_(604).pdf
Processing: BATCH_-_(101).pdf
Processing: BATCH_-_(602).pdf
Processing: BATCH_-_(702).pdf
Processing: BATCH_-_(703).pdf
Processing: BATCH_-_(404).pdf
Processing: BATCH_-_(504).pdf
Processing: BATCH_-_(603).pdf
Processing: BATCH_-_(505).pdf
Processing: BATCH_-_(705).pdf
Processing: BATCH_-_(402).pdf
Processing: BATCH_-_(203).pdf
Processing: BATCH_-_(305).pdf
Processing: BATCH_-_(501).pdf
Processing: BATCH_-_(104).pdf
Processing: BATCH_-_(601).pdf
Processing: BATCH_-_(304).pdf
Processing: BATCH_-_(605).pdf
Processing: BATCH_-_(704).pdf
Processing: BATCH_-_(701).pdf
Processing: BATCH_-_(303).pdf
Processing: BATCH_-_(405).pdf
Processing: BATCH_-_(401).pdf
Processing: BATCH_-_(105).pdf
Processing: BATCH_-_(502).pdf
Processing: BATCH_-_(302).pdf
Processing: BATCH_-_(301).pdf
Processing: BATCH_-_(205).pdf
Processing: BATCH_-_(201).pdf
Processing: BATCH_-_(204).pdf
Processing: BATCH_-_(403).pdf
Processing: BATCH_-_(202).pdf
Processing: BATCH_-_(503).pdf
Processing